# Case Bank Manager
Manage the `planning_case_bank` Qdrant collection for few-shot planning examples.

**Cells:**
1. Setup & imports
2. Init collection
3. Delete collection
4. Batch upsert from CSV

In [5]:
import os
import json
import uuid
import pandas as pd
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PayloadSchemaType,
    PointStruct, Filter, FieldCondition, MatchValue
)
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from tqdm import tqdm

load_dotenv(override=True)

QDRANT_URL = os.getenv('QDRANT_URL')
QDRANT_API_KEY = os.getenv('QDRANT_API_KEY')
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
COLLECTION_NAME = os.getenv('QDRANT_CASE_BANK_COLLECTION_NAME', 'planning_case_bank')
CSV_PATH = os.path.join(os.getcwd(), 'planning_fewshot_examples.csv')

# Strip :6333 port from HTTPS cloud URL (Qdrant Cloud uses port 443)
_qdrant_url = QDRANT_URL.rstrip('/')
if _qdrant_url.startswith('https://') and _qdrant_url.endswith(':6333'):
    _qdrant_url = _qdrant_url[:-5]

client = QdrantClient(url=_qdrant_url, api_key=QDRANT_API_KEY)
embeddings = GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-001',
    output_dimensionality=768,
    google_api_key=GOOGLE_API_KEY,
    task_type='RETRIEVAL_DOCUMENT'
)
print(f'✅ Connected to Qdrant: {_qdrant_url}')
print(f'✅ Collection: {COLLECTION_NAME}')

python-dotenv could not parse statement starting at line 45


✅ Connected to Qdrant: https://c1bc0a64-3295-4203-8d2a-9a58ebfe80f8.sa-east-1-0.aws.cloud.qdrant.io
✅ Collection: Soccer_Case_Bank


In [3]:
# ── Cell 2: Init collection ──────────────────────────────────────────────────
existing = [c.name for c in client.get_collections().collections]
if COLLECTION_NAME in existing:
    print(f'⚠️  Collection "{COLLECTION_NAME}" already exists. Skipping creation.')
else:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE),
    )
    # Create payload indexes for fast filtered search
    client.create_payload_index(COLLECTION_NAME, 'has_media', PayloadSchemaType.BOOL)
    client.create_payload_index(COLLECTION_NAME, 'label', PayloadSchemaType.KEYWORD)
    client.create_payload_index(COLLECTION_NAME, 'use_case', PayloadSchemaType.KEYWORD)
    print(f'✅ Collection "{COLLECTION_NAME}" created with indexes: has_media, label, use_case')

✅ Collection "Soccer_Case_Bank" created with indexes: has_media, label, use_case


In [2]:
# ── Cell 3: Delete collection ────────────────────────────────────────────────
confirm = input(f'Type DELETE to confirm deletion of "{COLLECTION_NAME}": ')
if confirm.strip() == 'DELETE':
    client.delete_collection(COLLECTION_NAME)
    print(f'🗑️  Collection "{COLLECTION_NAME}" deleted.')
else:
    print('Aborted.')

🗑️  Collection "Soccer_Case_Bank" deleted.


In [6]:
# ── Cell 4: Batch upsert from CSV ────────────────────────────────────────────
BATCH_SIZE = 32

df = pd.read_csv(CSV_PATH)
print(f'📄 Loaded {len(df)} rows from CSV')

rows = df.to_dict('records')
total_upserted = 0

for batch_start in tqdm(range(0, len(rows), BATCH_SIZE), desc='Upserting batches'):
    batch = rows[batch_start: batch_start + BATCH_SIZE]
    texts = [str(r['query']) for r in batch]
    vectors = embeddings.embed_documents(texts)

    points = []
    for row, vector in zip(batch, vectors):
        payload = {
            'query': str(row['query']),
            'has_media': bool(row['has_media']),
            'label': str(row['label']),
            'use_case': str(row['use_case']),
            'tool_chains': str(row['tool_chains']),
            'sub_queries': str(row['sub_queries']),
            'reasoning': str(row['reasoning']),
        }
        points.append(PointStruct(id=str(uuid.uuid4()), vector=vector, payload=payload))

    client.upsert(collection_name=COLLECTION_NAME, points=points)
    total_upserted += len(points)

print(f'✅ Upserted {total_upserted} points into "{COLLECTION_NAME}"')
info = client.get_collection(COLLECTION_NAME)
print(f'📊 Collection points count: {info.points_count}')

📄 Loaded 84 rows from CSV


Upserting batches: 100%|██████████| 3/3 [00:09<00:00,  3.22s/it]


✅ Upserted 84 points into "Soccer_Case_Bank"
📊 Collection points count: 84


In [4]:
info.points_count

84